# Passo 2 (controllo del 26B) e Passo 3b — esecuzione e stato della generazione

Notebook di esecuzione delle generazioni con **tutte le scelte già fatte** (docs/decisions.md, voci 118-132). Le
generazioni girano **in background** (`experiments/run_batch.py`): il notebook si può chiudere e riaprire, la cella
*Stato* mostra a che punto è la generazione; una run interrotta si riprende rilanciando la stessa cella (`--resume`).

| | Passo 2 — controllo del 26B (voci 131-132) | Passo 3b — esperimento principale (voce 132) |
|---|---|---|
| esercizi | 20 di sviluppo | **54 di valutazione** (34 corpus + 20 De Bari) |
| condizioni | bm25 k = 3 | zero_shot, static, random, bm25 k = 2 e 3, dense, hybrid |
| ripetizioni | 2 | 3 |
| generazioni | 80 | 2.262 |
| config | `experiments/configs/dev_g26.yaml` | `experiments/configs/passo3b.yaml` |
| analisi | `experiments/analyze_g26.py` (regola registrata) | da scrivere e testare PRIMA delle run |

Scelte fisse: istruzioni v5 (PlantUML) e v5 + esempi indentati (compatto); esempi dal file congelato
`data/retrieval_frozen/retrievals_3b.jsonl`; random con seme per esercizio e ripetizione; temperatura 0,3, top_p
0,95, top_k 64; contesto 32768, max_tokens 4096; ragionamento spento (`thinking: none`).

**Sicurezze**: tutti gli interruttori sono a `False`. Il 3b resta bloccato finché i relatori non rispondono (domande
8, 11, 14, 16) e finché lo script di analisi del 3b non è scritto e testato.

In [ ]:
# ---- interruttori e parametri (lasciare False salvo decisione esplicita) ----
MODELLO = "G26"          # "G26" = Gemma 4 26B A4B QAT (se il passo 2 lo adotta) | "G12" = Gemma 4 12B QAT
CARICA_MODELLO = False   # carica il modello in LM Studio con il comando registrato (scarica quello caricato)
ESEGUI_PASSO2 = True     # avvia le 80 generazioni del controllo del 26B (voce 132)
SBLOCCA_3B = False       # avvia il 3b: SOLO dopo le risposte dei relatori e con lo script di analisi del 3b pronto

import json, os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # percorsi dei config e degli script relativi alla radice del repo
sys.path[:0] = [str(ROOT / "experiments"), str(ROOT / "generation"), str(ROOT / "retrieval"), str(ROOT / "corpus")]
import progress, run_batch

MODELS = {"G26": ("google/gemma-4-26b-a4b-qat", "--context-length 32768 --gpu 0.5 -y"),
          "G12": ("google/gemma-4-12b-qat", "--context-length 32768 --gpu max -y")}
PASSO2 = ("passo2_g26", [("experiments/configs/dev_g26.yaml", "P-G"), ("experiments/configs/dev_g26.yaml", "C-G")])
PASSO3B = (f"passo3b_{MODELLO.lower()}", [("experiments/configs/passo3b.yaml", f"{f}-{s}-{MODELLO}")
                                          for f in "PC" for s in ("corpus", "testset")])

def sh(cmd):
    """Esegue uno script del repo con lo stesso Python del notebook e ne stampa l'uscita."""
    r = subprocess.run([sys.executable, *cmd.split()], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(r.stdout[-6000:], r.stderr[-3000:] if r.returncode else "", f"[codice {r.returncode}]")
    return r.returncode

print("Python:", sys.executable)
print("Modello scelto:", MODELLO, MODELS[MODELLO][0])

## 1. Controlli preliminari
Ambiente, file congelato (sha256), ruoli congelati (tag `evalset-v1`), GPU, modello caricato in LM Studio con il
contesto giusto. Nessuna generazione.

In [ ]:
import urllib.request
ok = True
if ".venv" not in sys.executable:
    print("ATTENZIONE: il kernel non usa .venv (docs/STATUS.md, 'Ambiente della tesi'): le run del 3b vanno fatte lì")
from prompt_builder import PromptBuilder
b = PromptBuilder(retrieval_source="frozen")
print("file congelato: sha256", b.frozen_sha256[:16], "verificato,", len(b.frozen), "righe")
same = subprocess.run(["git", "diff", "--quiet", "evalset-v1", "--", "corpus/roles.yaml"], cwd=ROOT).returncode == 0
print("roles.yaml invariato da evalset-v1:", same); ok &= same
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
try:
    models = json.load(urllib.request.urlopen("http://localhost:1234/api/v1/models", timeout=10))
    loaded = {i["id"]: i["config"]["context_length"] for m in models.get("models", models.get("data", []))
              for i in m.get("loaded_instances", [])}
    want = MODELS[MODELLO][0]
    print("LM Studio, modelli caricati:", loaded)
    if loaded.get(want) != 32768:
        print(f"ATTENZIONE: serve {want} con contesto 32768 (cella 2)"); ok = False
except OSError as e:
    print("LM Studio non raggiungibile:", e); ok = False
print("\nCONTROLLI", "SUPERATI" if ok else "NON superati: non avviare le generazioni")

## 2. Caricare il modello in LM Studio
Comandi registrati: 26B con metà degli strati sulla GPU (voce 131), 12B interamente sulla GPU. Scarica i modelli
caricati. Con `CARICA_MODELLO = False` stampa soltanto il comando.

In [ ]:
key, opts = MODELS[MODELLO]
cmd = f"lms load {key} {opts}"
print("comando:", cmd)
if CARICA_MODELLO:
    print(subprocess.run("lms unload --all", shell=True, capture_output=True, text=True).stdout)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout[-1500:])

## 3. Passo 2 — controllo del 26B sul dev set (80 generazioni, circa 40 minuti)
Regola registrata PRIMA della run (voce 132): il 26B diventa il modello del 3b se, in entrambi i formati, dR ≥ −0,03,
dVc ≥ −2, dM_primaria ≥ −0,03, meno di 2 troncamenti, e la stima del 3b è ≤ 30 ore. Richiede il 26B caricato.

In [ ]:
nome, runs = PASSO2
if ESEGUI_PASSO2:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("ESEGUI_PASSO2 = False: nessuna generazione avviata")

### Stato del passo 2 (rieseguire la cella quando si vuole; `watch` aggiorna da solo ogni minuto)

In [ ]:
try:
    print(progress.report(PASSO2[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(PASSO2[0], every_s=60, max_minutes=60)   # aggiornamento automatico

### Analisi del passo 2 (a run complete): applica la regola e scrive `data/results/generation/dev_g26_analysis/summary.md`

In [ ]:
try:
    st = progress.status(PASSO2[0])
except FileNotFoundError:
    st = None
if st and st["fatte"] == st["attese"] and not st["vivo"]:
    sh("experiments/analyze_g26.py")
else:
    print("passo 2 non completo: l'analisi si esegue a run finite", f"({st['fatte']}/{st['attese']})" if st else "")

## 4. Passo 3b — esperimento principale (2.262 generazioni)
Quattro run per il modello scelto: `passo3b__{P,C}-{corpus,testset}-<modello>`. Prima un **dry run** (nessuna chiamata
al modello: lunghezze dei prompt e finestre di contesto), poi l'avvio, che richiede `SBLOCCA_3B = True`.

In [ ]:
nome, runs = PASSO3B
print("run del 3b:", [k for _, k in runs])
print("generazioni attese:", sum(progress.expected_calls(c, k) for c, k in runs))
# dry run (nessuna chiamata al modello), scritto in data/results/generation/dry_run/ (non versionato)
for c, k in runs:
    sh(f"experiments/run_experiment.py {c} --configuration {k} --dry-run")

In [ ]:
if SBLOCCA_3B:
    pid = run_batch.launch(nome, runs)
    print(f"batch {nome} avviato in background (pid {pid}); log: data/results/batch/{nome}/batch.log")
else:
    print("SBLOCCA_3B = False: il 3b non parte (relatori, domande 8, 11, 14, 16; script di analisi del 3b)")

### Stato del 3b

In [ ]:
try:
    print(progress.report(PASSO3B[0]))
except FileNotFoundError:
    print("batch non ancora avviato")
# progress.watch(PASSO3B[0], every_s=120, max_minutes=120)   # aggiornamento automatico

## 5. Fermare un batch
`run_batch.stop(nome)` ferma **alla fine della run in corso**; `run_batch.stop(nome, now_=True)` termina subito il
processo. In entrambi i casi le chiamate già fatte restano (cache e manifest): rilanciando la cella di avvio il batch
riprende dalle mancanti.

In [ ]:
# run_batch.stop(PASSO2[0])
# run_batch.stop(PASSO3B[0], now_=True)

## 6. Confronto visivo di un esercizio: ground truth, PlantUML e JSON compatto
Per l'esercizio e la ripetizione scelti: il **diagramma di riferimento** e le **risposte delle run indicate**, una accanto
all'altra, disegnate tutte allo stesso modo (PlantUML canonico → SVG con `plantuml.jar` locale; classi **verdi** = presenti
nel GT, **arancioni** = in più). Per ogni risposta: livello di validità, V / Vc / J / R / M e l'elenco delle relazioni
(corretta, tipo sbagliato, direzione invertita, molteplicità diverse, mancante, in più); le risposte non valide mostrano
il testo grezzo. La pagina si salva anche in `data/results/generation/confronti/` per aprirla nel browser.

- Passo 2: `RUNS = ["dev_g26__P-G", "dev_g26__C-G"]`; per confrontare anche il 12B aggiungere `"dev_v5__P-G"` e
  `"dev_c5indent__C-G"`.
- 3b: per esempio `RUNS = ["passo3b__P-corpus-G26", "passo3b__C-corpus-G26"]` con `CONDIZIONE = "bm25"` e `K = 3`
  (senza condizione compaiono tutte le condizioni dell'esercizio).

In [ ]:
import compare_view as cv
from IPython.display import display

RUNS = ["dev_g26__P-G", "dev_g26__C-G"]   # + "dev_v5__P-G", "dev_c5indent__C-G" per il confronto con il 12B
ESERCIZIO = "Boeing"
RIPETIZIONE = 0
CONDIZIONE, K = None, None                 # nel 3b: es. "bm25", 3

print("esercizi disponibili:", ", ".join(cv.exercises(RUNS)))
display(cv.show(ESERCIZIO, RUNS, RIPETIZIONE, CONDIZIONE, K))

## 7. Tutti i risultati di una run: report di revisione completo
Pagina HTML con **tutti** gli esercizi della run: un indice con V / Vc / J / R / M per esercizio e condizione, poi per
ogni esercizio il ground truth e tutte le risposte (tutte le ripetizioni e i formati), con le relazioni classificate.
Si apre nel browser predefinito. Salvata in `data/results/generation/<nome>_review/index.html` (non versionata).
Le metriche complessive e l'esito della regola restano nel `summary.md` dell'analisi (cella del passo 2).

In [ ]:
import os
REPORT_RUNS = ["dev_g26__P-G", "dev_g26__C-G"]   # + "dev_v5__P-G", "dev_c5indent__C-G" per affiancare il 12B
REPORT_NOME = "dev_g26"                          # es. "dev_g26_vs_12b" con le quattro run
REPORT_FILTRI = []                               # es. ["rep=0"], ["q=Boeing"], nel 3b ["cond=bm25", "k=3"]

args = " ".join(["experiments/review_report.py --runs", *REPORT_RUNS, "--name", REPORT_NOME,
                 *[f"--filter {f}" for f in REPORT_FILTRI]])
if sh(args) == 0:
    page = ROOT / "data" / "results" / "generation" / f"{REPORT_NOME}_review" / "index.html"
    print("aperta nel browser:", page)
    os.startfile(page)   # Windows: browser predefinito

## 8. Dopo le run
- Passo 2: cella di analisi qui sopra; esito da registrare in docs/decisions.md (e `MODELLO` di conseguenza).
- Passo 3b: lo script di analisi (confronti appaiati sui 54 e sui 52, static contro retrieval su 53, pertinenza e
  qualità tra esercizi ed entro esercizio; voci 129-130) va scritto e testato PRIMA di avviare le run.
- Revisione visiva di un campione: `python experiments/review_report.py --runs <run> --name <nome>`.